# Migração do ConvAI2 BothRevised sem candidatos

Este notebook baixa o pacote oficial do ParlAI, converte somente `convai2:BothRevised:no_cands` para uma conversa por linha e, opcionalmente, publica a configuração `both_revised_no_candidates` no Hugging Face Hub.

Dependências: `datasets` e `huggingface_hub` (disponíveis, por exemplo, no extra `prompt-enhancing` deste projeto). O ParlAI não é necessário.

In [9]:
from __future__ import annotations

import hashlib
import json
import os
import tarfile
import urllib.request
from pathlib import Path
from typing import Any

from datasets import Dataset, DatasetDict, load_dataset

HUB_REPO_ID = "visual-memory/ConvAI2"
PUSH_TO_HUB = False
CONFIG_NAME = "both_revised_no_candidates"

SOURCE_URL = "http://parl.ai/downloads/convai2/convai2_fix_723.tgz"
SOURCE_SHA256 = "d0ae89defe2fd0b0a4221eaa642a457d7d40cef475f54798119c7f3b8dd9361d"
CACHE_DIR = Path(
    os.environ.get(
        "CONVAI2_CACHE_DIR",
        Path.home() / ".cache" / "convai2-huggingface-migration",
    )
)
ARCHIVE_PATH = CACHE_DIR / "convai2_fix_723.tgz"

SOURCE_FILES = {
    "train": "train_both_revised_no_cands.txt",
    "validation": "valid_both_revised_no_cands.txt",
}


## Download e verificação

O download é gravado primeiro em um arquivo parcial e só substitui o cache depois da validação do SHA-256 oficial.

In [2]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_archive(path: Path) -> None:
    actual_sha256 = sha256_file(path)
    if actual_sha256 != SOURCE_SHA256:
        raise ValueError(
            f"Checksum inválido para {path}: esperado {SOURCE_SHA256}, "
            f"recebido {actual_sha256}."
        )


def download_archive() -> Path:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if ARCHIVE_PATH.exists():
        try:
            validate_archive(ARCHIVE_PATH)
            print(f"Usando arquivo validado em cache: {ARCHIVE_PATH}")
            return ARCHIVE_PATH
        except ValueError:
            print("O arquivo em cache está incompleto ou corrompido; baixando novamente.")

    partial_path = ARCHIVE_PATH.with_suffix(ARCHIVE_PATH.suffix + ".part")
    request = urllib.request.Request(
        SOURCE_URL, headers={"User-Agent": "convai2-huggingface-migration/1.0"}
    )
    print(f"Baixando {SOURCE_URL} ...")
    try:
        with urllib.request.urlopen(request) as response, partial_path.open("wb") as output:
            while chunk := response.read(1024 * 1024):
                output.write(chunk)
        validate_archive(partial_path)
        partial_path.replace(ARCHIVE_PATH)
    except Exception:
        partial_path.unlink(missing_ok=True)
        raise

    print(f"Download concluído e validado: {ARCHIVE_PATH}")
    return ARCHIVE_PATH


archive_path = download_archive()


Usando arquivo validado em cache: /home/aluno_paulosantana/.cache/convai2-huggingface-migration/convai2_fix_723.tgz


## Leitura e conversão

No formato FBDialog usado pelo ParlAI, cada episódio reinicia a numeração em `1`. As primeiras linhas contêm `your persona:` ou `partner's persona:` e cada linha posterior contém exatamente `text\tresponse`.

In [3]:
YOUR_PERSONA_PREFIX = "your persona: "
PARTNER_PERSONA_PREFIX = "partner's persona: "
PERSONA_PREFIXES = {
    YOUR_PERSONA_PREFIX: "your_persona",
    PARTNER_PERSONA_PREFIX: "partner_persona",
}
EXPECTED_COLUMNS = [
    "conversation_id",
    "your_persona",
    "partner_persona",
    "turns",
    "source_split",
]


def read_source_files(archive_path: Path) -> dict[str, str]:
    expected_names = set(SOURCE_FILES.values())
    contents: dict[str, str] = {}

    with tarfile.open(archive_path, mode="r:gz") as archive:
        members_by_name = {
            member.name.removeprefix("./"): member
            for member in archive.getmembers()
            if member.isfile()
        }
        missing = sorted(expected_names - set(members_by_name))
        if missing:
            raise FileNotFoundError(f"Arquivos esperados ausentes no pacote: {missing}")

        for hf_split, source_name in SOURCE_FILES.items():
            extracted = archive.extractfile(members_by_name[source_name])
            if extracted is None:
                raise OSError(f"Não foi possível ler {source_name} do pacote.")
            contents[hf_split] = extracted.read().decode("utf-8")

    return contents


def make_conversation_id(source_split: str, conversation_index: int) -> str:
    identity = f"{CONFIG_NAME}\0{source_split}\0{conversation_index}"
    return hashlib.sha256(identity.encode("utf-8")).hexdigest()


def parse_split(source_text: str, *, source_split: str) -> list[dict[str, Any]]:
    conversations: list[dict[str, Any]] = []
    your_persona: list[str] = []
    partner_persona: list[str] = []
    turns: list[dict[str, Any]] = []
    expected_line_number = 1

    def finish_conversation() -> None:
        nonlocal your_persona, partner_persona, turns
        if not your_persona and not partner_persona and not turns:
            return
        conversation_index = len(conversations)
        if not your_persona:
            raise ValueError(
                f"your_persona vazia em split={source_split!r}, "
                f"conversa={conversation_index}."
            )
        if not partner_persona:
            raise ValueError(
                f"partner_persona vazia em split={source_split!r}, "
                f"conversa={conversation_index}."
            )
        if not turns:
            raise ValueError(
                f"Conversa sem turnos em split={source_split!r}, "
                f"conversa={conversation_index}."
            )
        conversations.append(
            {
                "conversation_id": make_conversation_id(
                    source_split, conversation_index
                ),
                "your_persona": your_persona,
                "partner_persona": partner_persona,
                "turns": turns,
                "source_split": source_split,
            }
        )
        your_persona = []
        partner_persona = []
        turns = []

    for physical_line_number, raw_line in enumerate(source_text.splitlines(), start=1):
        if not raw_line.strip():
            raise ValueError(
                f"Linha vazia em split={source_split!r}, linha={physical_line_number}."
            )
        try:
            number_text, payload = raw_line.split(" ", maxsplit=1)
            source_line_number = int(number_text)
        except (ValueError, TypeError) as exc:
            raise ValueError(
                f"Linha ParlAI malformada em split={source_split!r}, "
                f"linha={physical_line_number}: {raw_line!r}."
            ) from exc

        if source_line_number == 1 and (your_persona or partner_persona or turns):
            finish_conversation()
            expected_line_number = 1
        if source_line_number != expected_line_number:
            raise ValueError(
                f"Numeração inválida em split={source_split!r}, "
                f"linha={physical_line_number}: esperado {expected_line_number}, "
                f"recebido {source_line_number}."
            )
        expected_line_number += 1

        matching_prefix = next(
            (prefix for prefix in PERSONA_PREFIXES if payload.startswith(prefix)),
            None,
        )
        if matching_prefix is not None:
            if turns:
                raise ValueError(
                    f"Persona encontrada depois do início do diálogo em "
                    f"split={source_split!r}, linha={physical_line_number}."
                )
            statement = payload.removeprefix(matching_prefix).strip()
            if not statement:
                raise ValueError(
                    f"Frase de persona vazia em split={source_split!r}, "
                    f"linha={physical_line_number}."
                )
            target = (
                your_persona
                if matching_prefix == YOUR_PERSONA_PREFIX
                else partner_persona
            )
            target.append(statement)
            continue

        if "persona:" in payload.lower():
            raise ValueError(
                f"Prefixo de persona desconhecido em split={source_split!r}, "
                f"linha={physical_line_number}: {payload!r}."
            )

        fields = payload.split("\t")
        if len(fields) != 2:
            raise ValueError(
                f"Esperadas exatamente duas colunas (text e response), sem "
                f"candidatos, em split={source_split!r}, linha={physical_line_number}; "
                f"recebidas {len(fields)}."
            )
        text, response = (field.strip() for field in fields)
        if not text or not response:
            raise ValueError(
                f"Text ou response vazio em split={source_split!r}, "
                f"linha={physical_line_number}."
            )
        turns.append(
            {"turn_id": len(turns), "text": text, "response": response}
        )

    finish_conversation()
    return conversations


source_texts = read_source_files(archive_path)


## Estrutura original do ParlAI

Os arquivos não possuem cabeçalho. O ParlAI usa o formato posicional FBDialog:

| Elemento | Presença | Significado |
|---|---|---|
| `episode_line` | Todas as linhas | Número antes do primeiro espaço. Começa em `1` e cresce dentro do diálogo; um novo `1` inicia outro diálogo. |
| `text` | Todas as linhas | Conteúdo depois do número. Nas primeiras linhas, começa com `your persona:` ou `partner's persona:`; nas demais, é a fala de entrada. |
| `label`/`response` | Somente linhas de diálogo | Resposta correta, separada de `text` por uma tabulação (`\t`). |
| `reward` | Ausente nesta variante | O formato FBDialog aceita reward, mas os arquivos selecionados não o fornecem. |
| `label_candidates` | Ausente nesta variante | Não existe porque usamos `BothRevised:no_cands`. |

Portanto, uma linha de persona tem `episode_line + text`; uma linha de turno tem `episode_line + text + response`. Os dois prefixos de persona identificam participantes diferentes.

In [4]:
def describe_original_split(source_text: str, *, source_split: str) -> None:
    lines = source_text.splitlines()
    episode_count = 0
    your_persona_line_count = 0
    partner_persona_line_count = 0
    dialogue_line_count = 0
    first_your_persona_line: str | None = None
    first_partner_persona_line: str | None = None
    first_dialogue_line: str | None = None

    for raw_line in lines:
        number_text, payload = raw_line.split(" ", maxsplit=1)
        if number_text == "1":
            episode_count += 1
        if payload.startswith(YOUR_PERSONA_PREFIX):
            your_persona_line_count += 1
            first_your_persona_line = first_your_persona_line or raw_line
        elif payload.startswith(PARTNER_PERSONA_PREFIX):
            partner_persona_line_count += 1
            first_partner_persona_line = first_partner_persona_line or raw_line
        else:
            dialogue_line_count += 1
            first_dialogue_line = first_dialogue_line or raw_line

    print(f"Arquivo: {SOURCE_FILES[source_split]}")
    print(f"Split de origem: {source_split}")
    print(f"Linhas físicas: {len(lines):,}")
    print(f"Episódios (reinícios em 1): {episode_count:,}")
    print(f"Linhas de your_persona (1 campo): {your_persona_line_count:,}")
    print(
        f"Linhas de partner_persona (1 campo): "
        f"{partner_persona_line_count:,}"
    )
    print(f"Linhas de diálogo (2 campos): {dialogue_line_count:,}")
    print("Schema de your_persona: <episode_line> your persona: <statement>")
    print(
        "Schema de partner_persona: "
        "<episode_line> partner's persona: <statement>"
    )
    print("Schema da linha de diálogo: <episode_line> <text>\t<response>")
    print(f"Exemplo de your_persona (raw): {first_your_persona_line!r}")
    print(f"Exemplo de partner_persona (raw): {first_partner_persona_line!r}")
    print(f"Exemplo de diálogo (raw): {first_dialogue_line!r}")


print("ESTRUTURA ORIGINAL DO PARLAI")
print("Formato: FBDialog; variante: convai2:BothRevised:no_cands")
print(
    "Campos condicionais por linha: your_persona, partner_persona "
    "OU (text, response)"
)
for split, source_text in source_texts.items():
    print("\n" + "-" * 80)
    describe_original_split(source_text, source_split=split)


ESTRUTURA ORIGINAL DO PARLAI
Formato: FBDialog; variante: convai2:BothRevised:no_cands
Campos condicionais por linha: your_persona, partner_persona OU (text, response)

--------------------------------------------------------------------------------
Arquivo: train_both_revised_no_cands.txt
Split de origem: train
Linhas físicas: 292,190
Episódios (reinícios em 1): 17,878
Linhas de your_persona (1 campo): 80,365
Linhas de partner_persona (1 campo): 80,387
Linhas de diálogo (2 campos): 131,438
Schema de your_persona: <episode_line> your persona: <statement>
Schema de partner_persona: <episode_line> partner's persona: <statement>
Schema da linha de diálogo: <episode_line> <text>	<response>
Exemplo de your_persona (raw): '1 your persona: i love to redesign houses.'
Exemplo de partner_persona (raw): "5 partner's persona: my favorite hobbies are based on old fashioned life skills."
Exemplo de diálogo (raw): "9 hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in sh

### Exemplo detalhado: um diálogo no arquivo original

A célula abaixo seleciona o primeiro diálogo de treino e mostra todas as **N linhas** que o compõem. A fronteira é determinada pelo próximo `episode_line = 1`, não por uma linha vazia.

In [5]:
def get_first_episode_lines(source_text: str) -> list[str]:
    episode_lines: list[str] = []
    for raw_line in source_text.splitlines():
        number_text, _ = raw_line.split(" ", maxsplit=1)
        if number_text == "1" and episode_lines:
            break
        episode_lines.append(raw_line)
    return episode_lines


example_raw_lines = get_first_episode_lines(source_texts["train"])
print(f"Diálogo escolhido: primeiro episódio de {SOURCE_FILES['train']}")
print(f"N = {len(example_raw_lines)} linhas originais\n")

for physical_index, raw_line in enumerate(example_raw_lines, start=1):
    number_text, payload = raw_line.split(" ", maxsplit=1)
    visible_raw_line = raw_line.replace("\t", " <TAB> ")
    print(f"Linha {physical_index:02d} (raw): {visible_raw_line}")
    matching_prefix = next(
        (prefix for prefix in PERSONA_PREFIXES if payload.startswith(prefix)),
        None,
    )
    if matching_prefix is not None:
        statement = payload.removeprefix(matching_prefix)
        target_column = PERSONA_PREFIXES[matching_prefix]
        print(f"  tipo={target_column}")
        print(f"  episode_line={number_text}; text={payload!r}")
        print(
            f"  ação: acrescentar {statement!r} à lista "
            f"{target_column}\n"
        )
    else:
        text, response = payload.split("\t")
        print("  tipo=turno")
        print(f"  episode_line={number_text}; text={text!r}; response={response!r}")
        print("  ação: criar um turno ligando esta entrada à resposta correta\n")


Diálogo escolhido: primeiro episódio de train_both_revised_no_cands.txt
N = 15 linhas originais

Linha 01 (raw): 1 your persona: i love to redesign houses.
  tipo=your_persona
  episode_line=1; text='your persona: i love to redesign houses.'
  ação: acrescentar 'i love to redesign houses.' à lista your_persona

Linha 02 (raw): 2 your persona: killing for sport is my hobby.
  tipo=your_persona
  episode_line=2; text='your persona: killing for sport is my hobby.'
  ação: acrescentar 'killing for sport is my hobby.' à lista your_persona

Linha 03 (raw): 3 your persona: i shot an arrow the other day !.
  tipo=your_persona
  episode_line=3; text='your persona: i shot an arrow the other day !.'
  ação: acrescentar 'i shot an arrow the other day !.' à lista your_persona

Linha 04 (raw): 4 your persona: i like to get dressed up.
  tipo=your_persona
  episode_line=4; text='your persona: i like to get dressed up.'
  ação: acrescentar 'i like to get dressed up.' à lista your_persona

Linha 05 (ra

#### Como essas linhas formam o diálogo

1. As linhas iniciadas por `your persona:` descrevem a persona revisada do agente que produz `response`; elas são acumuladas em `your_persona`.
2. As linhas iniciadas por `partner's persona:` descrevem a persona revisada do interlocutor que produz `text`; elas são acumuladas em `partner_persona`.
3. A primeira linha sem um desses prefixos encerra a seção de personas e inicia os turnos.
4. Em cada turno, `text` é a fala do parceiro e `response` é a resposta correta do agente identificado por `your_persona`. A ordem determina `turn_id = 0, 1, ...`.
5. Todas as linhas pertencem à mesma conversa até a numeração reiniciar em `1`; a linha `1` seguinte já pertence ao próximo registro.
6. Não há candidates nem reward para transportar nesta variante.

## Dataset recriado no formato Hugging Face

Agora os episódios originais são agrupados em uma linha por conversa, com as frases das duas personas e os turnos representados como listas estruturadas.

In [6]:
parsed_rows = {
    split: parse_split(text, source_split=split)
    for split, text in source_texts.items()
}
dataset_dict = DatasetDict(
    {split: Dataset.from_list(rows) for split, rows in parsed_rows.items()}
)

for split, dataset in dataset_dict.items():
    print(f"{split}: {len(dataset):,} conversas")
print(f"Colunas: {dataset_dict['train'].column_names}")


train: 17,878 conversas
validation: 1,000 conversas
Colunas: ['conversation_id', 'your_persona', 'partner_persona', 'turns', 'source_split']


### O mesmo diálogo depois da conversão

As N linhas mostradas acima tornam-se o registro único abaixo. `conversation_id` identifica deterministicamente a configuração, o split e a posição original do diálogo.

In [7]:
converted_example = parsed_rows["train"][0]
assert (
    len(converted_example["your_persona"])
    + len(converted_example["partner_persona"])
    + len(converted_example["turns"])
) == len(example_raw_lines)
print("Schema convertido:")
print(dataset_dict["train"].features)
print("\nRegistro convertido (uma linha por diálogo):")
print(json.dumps(converted_example, ensure_ascii=False, indent=2))


Schema convertido:
{'conversation_id': Value('string'), 'your_persona': List(Value('string')), 'partner_persona': List(Value('string')), 'turns': List({'turn_id': Value('int64'), 'text': Value('string'), 'response': Value('string')}), 'source_split': Value('string')}

Registro convertido (uma linha por diálogo):
{
  "conversation_id": "c1b47f497e641797e4b29b21634d438c565429d48259fac720e8d42483fa20af",
  "your_persona": [
    "i love to redesign houses.",
    "killing for sport is my hobby.",
    "i shot an arrow the other day !.",
    "i like to get dressed up."
  ],
  "partner_persona": [
    "my favorite hobbies are based on old fashioned life skills.",
    "i race large felines who are in captivity to remain healthy.",
    "i was a really good runner when i was younger.",
    "i am a carnivore."
  ],
  "turns": [
    {
      "turn_id": 0,
      "text": "hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape .",
      "response": "you must be very fast

## Validações

In [8]:
assert list(dataset_dict) == ["train", "validation"]
assert all(ds.column_names == EXPECTED_COLUMNS for ds in dataset_dict.values())
assert all("candidates" not in ds.column_names for ds in dataset_dict.values())
assert dataset_dict["train"].features == dataset_dict["validation"].features

all_ids: set[str] = set()
for split, rows in parsed_rows.items():
    for conversation_index, row in enumerate(rows):
        expected_id = make_conversation_id(split, conversation_index)
        assert row["conversation_id"] == expected_id
        assert expected_id not in all_ids, f"ID duplicado: {expected_id}"
        all_ids.add(expected_id)
        assert row["source_split"] == split
        assert row["your_persona"] and all(
            item.strip() for item in row["your_persona"]
        )
        assert row["partner_persona"] and all(
            item.strip() for item in row["partner_persona"]
        )
        assert row["turns"]
        assert [turn["turn_id"] for turn in row["turns"]] == list(
            range(len(row["turns"]))
        )
        assert all(turn["text"].strip() for turn in row["turns"])
        assert all(turn["response"].strip() for turn in row["turns"])
        assert all(set(turn) == {"turn_id", "text", "response"} for turn in row["turns"])

# Uma segunda conversão comprova contagens, ordem e IDs determinísticos.
for split, source_text in source_texts.items():
    reparsed_rows = parse_split(source_text, source_split=split)
    assert reparsed_rows == parsed_rows[split]

print(f"Validações concluídas: {len(all_ids):,} IDs únicos.")


Validações concluídas: 18,878 IDs únicos.


## Publicação opcional

Revise a amostra e as contagens, preencha `HUB_REPO_ID` e altere `PUSH_TO_HUB` para `True`.

In [11]:
PUSH_TO_HUB = True

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [13]:
if PUSH_TO_HUB:
    if HUB_REPO_ID.startswith("SEU_USUARIO/"):
        raise ValueError("Preencha HUB_REPO_ID antes de publicar.")

    dataset_dict.push_to_hub(HUB_REPO_ID, config_name=CONFIG_NAME)

    published = load_dataset(HUB_REPO_ID, CONFIG_NAME)
    assert list(published) == list(dataset_dict)
    for split in dataset_dict:
        assert len(published[split]) == len(dataset_dict[split])
        assert published[split].column_names == dataset_dict[split].column_names
        assert published[split].features == dataset_dict[split].features
    print(f"Dataset publicado e verificado: {HUB_REPO_ID}/{CONFIG_NAME}")
else:
    print("Publicação desativada (PUSH_TO_HUB=False).")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

both_revised_no_candidates/train-00000-o(…):   0%|          | 0.00/21.5M [00:00<?, ?B/s]

both_revised_no_candidates/validation-00(…):   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset publicado e verificado: visual-memory/ConvAI2/both_revised_no_candidates
